In [19]:
import re
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama

from langchain_text_splitters import RecursiveCharacterTextSplitter

In [20]:
def load_pdfs(pdf_dir):
    documents = []
    pdf_dir = Path(pdf_dir)

    for pdf in pdf_dir.glob("**/*.pdf"):
        loader = PyPDFLoader(str(pdf))
        docs = loader.load()

        for d in docs:
            d.metadata["source"] = pdf.name

        documents.extend(docs)

    return documents


docs = load_pdfs("../data")
print("Pages loaded:", len(docs))

Pages loaded: 57


In [21]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)
print("Chunks:", len(chunks))

Chunks: 268


In [22]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(chunks, embedding_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 201.29it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
llm = Ollama(
    model="llama3",
    temperature=0
)

In [24]:
def retrieve_context_with_citations(query, vectorstore, k=4):
    docs = vectorstore.similarity_search(query, k=k)

    context_blocks = []
    citation_ids = []

    for i, doc in enumerate(docs):
        cid = f"Doc{i+1}"
        citation_ids.append(cid)

        block = f"[{cid}] {doc.page_content}"
        context_blocks.append(block)

    context = "\n\n".join(context_blocks)
    return context, citation_ids

In [25]:
def generate_answer_with_citations(query, context, citation_ids, llm):
    prompt = f"""
Answer the question using ONLY the context below.
Each factual statement MUST include a citation.

Allowed citation IDs: {', '.join(citation_ids)}

Context:
{context}

Question:
{query}

Answer (with citations):
"""
    return llm.invoke(prompt).strip()


In [26]:
def split_into_claims(answer):
    return [s.strip() for s in answer.split('.') if s.strip()]

In [27]:
def extract_citations(claim):
    return re.findall(r"\[Doc\d+\]", claim)

In [28]:
def citation_based_hallucination_check(answer, citation_ids):
    claims = split_into_claims(answer)

    supported = 0
    claim_results = []

    for claim in claims:
        citations = extract_citations(claim)

        valid = any(c in citation_ids for c in citations)

        if valid:
            supported += 1
            claim_results.append((claim, "SUPPORTED"))
        else:
            claim_results.append((claim, "NOT SUPPORTED"))

    total = len(claims)
    coverage = supported / total if total > 0 else 0

    if supported == total:
        verdict = "NOT HALLUCINATION"
    elif supported == 0:
        verdict = "HALLUCINATION"
    else:
        verdict = "PARTIAL HALLUCINATION"

    return verdict, coverage, claim_results


In [29]:
def citation_rag_pipeline(query, vectorstore, llm):
    context, citation_ids = retrieve_context_with_citations(query, vectorstore)

    answer = generate_answer_with_citations(
        query, context, citation_ids, llm
    )

    verdict, coverage, claim_results = citation_based_hallucination_check(
        answer, citation_ids
    )

    return {
        "question": query,
        "answer": answer,
        "hallucination_verdict": verdict,
        "citation_coverage": coverage,
        "claim_analysis": claim_results
    }

In [30]:
query = "What recent techniques have been developed to infer whether a large language model is telling the truth?"

result = citation_rag_pipeline(query, vectorstore, llm)

print("Question:", result["question"])
print("\nAnswer:\n", result["answer"])

print("\nClaim Analysis:")
for c in result["claim_analysis"]:
    print(c)

print("\nHallucination Verdict:", result["hallucination_verdict"])
print("Citation Coverage:", result["citation_coverage"])

Question: What recent techniques have been developed to infer whether a large language model is telling the truth?

Answer:
 According to Christiano et al. (2021), one approach to determine whether a large language model believes a statement to be true relies on inspecting model outputs, such as internal chain-of-thoughts like the example provided in Doc1. Additionally, recent work has developed techniques that train probes for classifying truthfulness based on the model's internal state when processing statements (Azaria & Mitchell, 2023; Li et al., 2023b; Burns et al., 2023).

Claim Analysis:
('According to Christiano et al', 'NOT SUPPORTED')
('(2021), one approach to determine whether a large language model believes a statement to be true relies on inspecting model outputs, such as internal chain-of-thoughts like the example provided in Doc1', 'NOT SUPPORTED')
("Additionally, recent work has developed techniques that train probes for classifying truthfulness based on the model's int